<a href="https://colab.research.google.com/github/DerWeiseTeufel/MVM_2025/blob/main/LLMsimilarity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import userdata


In [6]:
import pandas as pd
path = userdata.get('path_to_dataset')

# Предположим, у вас есть DataFrame с товарами
df_products = pd.read_csv(path)
df_products.head()

,NameFull
0,Куртка рабочая сигнальная летняя защитная от о...
1,Клапан регулирующий черт. 1850973 D75мм 25Х2М1Ф
2,Анкер химический (капсула с клеевым составом) ...
3,Анкер химический двухкомпонентный Fischer FIS ...
4,Анкер химический Himtex VESF PROFI 200 CAN2004...


In [ ]:
df_products = df_products.rename(columns={'NameFull':'description'})
df_products.head()

,description
0,Куртка рабочая сигнальная летняя защитная от о...
1,Клапан регулирующий черт. 1850973 D75мм 25Х2М1Ф
2,Анкер химический (капсула с клеевым составом) ...
3,Анкер химический двухкомпонентный Fischer FIS ...
4,Анкер химический Himtex VESF PROFI 200 CAN2004...


# Using sbert

In [ ]:
COL_NAME = 'NameFull'

In [ ]:
!pip install faiss-cpu  # Для CPU-версии

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 25.7 MB/s eta 0:00:00


In [ ]:
# create_faiss_index_sbert.py
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
from faiss import write_index, IndexFlatIP
import numpy as np

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
    sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
    return sum_embeddings / sum_mask

# 1. Загрузите данные
df = pd.read_csv(path)
texts = df[COL_NAME].tolist()[:100]

# 2. Загрузите SBERT модель
print("Загрузка модели sbert_large_mt_nlu_ru...")
tokenizer = AutoTokenizer.from_pretrained("sberbank-ai/sbert_large_mt_nlu_ru")
model = AutoModel.from_pretrained("sberbank-ai/sbert_large_mt_nlu_ru")
model.eval()

# 3. Создайте эмбеддинги
embeddings = []
batch_size = 8

print("Создание эмбеддингов...")
for i in range(0, len(texts), batch_size):
    batch = texts[i:i+batch_size]
    encoded = tokenizer(
        batch,
        padding=True,
        truncation=True,
        max_length=24,  # для SBERT
        return_tensors='pt'
    )

    with torch.no_grad():
        output = model(**encoded)
        batch_embeddings = mean_pooling(output, encoded['attention_mask'])
        embeddings.append(batch_embeddings.numpy())

# 4. Сохраните индекс
embeddings = np.vstack(embeddings)
index = IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
write_index(index, "subj_sbert_large_mt_nlu_ru_FAISS.index")

print(f"Индекс SBERT создан: {index.ntotal} векторов")

Загрузка модели sbert_large_mt_nlu_ru...
Создание эмбеддингов...
Индекс SBERT создан: 100 векторов


In [ ]:
# create_faiss_index.py
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from faiss import write_index, IndexFlatIP
import numpy as np

# Функция для pooling
def average_pool(last_hidden_states, attention_mask):
    last_hidden = last_hidden_states.masked_fill(~attention_mask[..., None].bool(), 0.0)
    return last_hidden.sum(dim=1) / attention_mask.sum(dim=1)[..., None]


# 2. Загрузите модель E5
print("Загрузка модели multilingual-e5-base...")
tokenizer = AutoTokenizer.from_pretrained('intfloat/multilingual-e5-base')
model = AutoModel.from_pretrained('intfloat/multilingual-e5-base')
model.eval()

# 3. Создайте эмбеддинги для всех текстов
embeddings = []
batch_size = 8  # уменьшите если не хватает памяти

print("Создание эмбеддингов...")
for i in range(0, len(texts), batch_size):
    batch = texts[i:i+batch_size]
    encoded = tokenizer(
        batch,
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors='pt'
    )

    with torch.no_grad():
        output = model(**encoded)
        batch_embeddings = average_pool(
            output.last_hidden_state,
            encoded['attention_mask']
        )
        batch_embeddings = F.normalize(batch_embeddings, p=2, dim=1)
        embeddings.append(batch_embeddings.numpy())

    if i % 100 == 0:
        print(f"Обработано {i}/{len(texts)} текстов")

# 4. Объедините все эмбеддинги
embeddings = np.vstack(embeddings)
print(f"Размерность эмбеддингов: {embeddings.shape}")

# 5. Создайте и сохраните FAISS индекс
print("Создание FAISS индекса...")
index = IndexFlatIP(embeddings.shape[1])  # IndexFlatIP для косинусного сходства
index.add(embeddings)
write_index(index, "subj_multilingual-e5-base_FAISS.index")

print(f"Индекс создан и сохранен в subj_multilingual-e5-base_FAISS.index")
print(f"Количество векторов: {index.ntotal}")

Загрузка модели multilingual-e5-base...
Создание эмбеддингов...
Обработано 0/100 текстов
Размерность эмбеддингов: (100, 768)
Создание FAISS индекса...
Индекс создан и сохранен в subj_multilingual-e5-base_FAISS.index
Количество векторов: 100


In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
from typing import List
import torch
import torch.nn.functional as F
from torch import Tensor
from transformers import AutoTokenizer, AutoModel
from faiss import read_index
import pandas as pd



# Read the dataset and prepare the subject pack
df = pd.read_csv(path)
subj_pack = df[COL_NAME].tolist()
id_pack = df.index.tolist()

# Pydantic model for the response
class SearchResult(BaseModel):
    model_name: str
    k_nearest: List[str]
    k_nearest_ids: List[int]

class SearchResponse(BaseModel):
    results: List[SearchResult]

# Load the tokenizer and model for multilingual-e5-base
tokenizer_e5 = AutoTokenizer.from_pretrained('intfloat/multilingual-e5-base')
model_e5 = AutoModel.from_pretrained('intfloat/multilingual-e5-base')
index_e5 = read_index("subj_multilingual-e5-base_FAISS.index")

# Load the tokenizer and model for sbert_large_mt_nlu_ru
tokenizer_sbert = AutoTokenizer.from_pretrained("sberbank-ai/sbert_large_mt_nlu_ru")
model_sbert = AutoModel.from_pretrained("sberbank-ai/sbert_large_mt_nlu_ru")
index_sbert = read_index("subj_sbert_large_mt_nlu_ru_FAISS.index")

# Function to perform average pooling on last hidden states
def average_pool(last_hidden_states: Tensor, attention_mask: Tensor) -> Tensor:
    last_hidden = last_hidden_states.masked_fill(~attention_mask[..., None].bool(), 0.0)
    return last_hidden.sum(dim=1) / attention_mask.sum(dim=1)[..., None]

#Mean Pooling - Take attention mask into account for correct averaging
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0] #First element of model_output contains all token embeddings
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
    sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
    return sum_embeddings / sum_mask

def search(text: str = '', k: int = 5):
    # e5
    encoded_input = tokenizer_e5(text, padding=True, truncation=True, max_length=512, return_tensors='pt')

    with torch.no_grad():
        model_output = model_e5(**encoded_input)
        query_vector = average_pool(model_output.last_hidden_state, encoded_input['attention_mask'])
        query_vector = F.normalize(query_vector, p=2, dim=1)

    # Search in e5 index
    D, I = index_e5.search(query_vector, k)
    results = [subj_pack[id] for id in I[0]]
    ids = [id_pack[id] for id in I[0]]
    e5_result = SearchResult(model_name='multilingual-e5-base', k_nearest=results, k_nearest_ids=ids)

    # Process query using sbert model
    encoded_input = tokenizer_sbert(text, padding=True, truncation=True, max_length=24, return_tensors='pt')
    with torch.no_grad():
        model_output = model_sbert(**encoded_input)
        query_vector = mean_pooling(model_output, encoded_input['attention_mask'])

    # Search in sbert index
    D, I = index_sbert.search(query_vector, k)
    results = [subj_pack[id] for id in I[0]]
    ids = [id_pack[id] for id in I[0]]
    sbert_result = SearchResult(model_name='sbert_large_mt_nlu_ru', k_nearest=results, k_nearest_ids=ids)
    # Return combined results from both models
    return SearchResponse(results=[e5_result, sbert_result])


In [ ]:
search("Ц15хр")

SearchResponse(results=[SearchResult(model_name='multilingual-e5-base', k_nearest=['Ключ динамометрический DREMOMETER 1500...3000Нм GEDORE 7717160', 'Ключ трубный рычажный КТР-0 21315016', 'Ключ гаечный двусторонний кольцевой Новтул 7811-0508 13х17 Ц15хр ГОСТ 2906-80', 'Клапан регулирующий черт. 1850973 D75мм 25Х2М1Ф', 'Ключ гаечный двусторонний кольцевой Новтул 7811-0288 14х17 Ц15хр ГОСТ 2906-80'], k_nearest_ids=[78, 76, 66, 1, 68]), SearchResult(model_name='sbert_large_mt_nlu_ru', k_nearest=['Клапан регулирующий черт. 1850973 D75мм 25Х2М1Ф', 'Кувалда 1кг Ермак 662-425', 'Кувалда 2кг Ермак 662-427', 'Ключ гаечный двусторонний с открытым зевом Новтул 7811-0026 24х27 Ц15хр ГОСТ 2839-80', 'Ключ гаечный двусторонний с открытым зевом Новтул 7811-0045 41х46 Ц15хр ГОСТ 2839-80'], k_nearest_ids=[1, 28, 27, 67, 57])])

## проблемы
*   Долго для 50к текстов: 50000 * 20 sec = 200 minutes - DONE, used GPU
*   Какой пуллинг использовать?
avg is ok
*   Какие еще модели есть для этого эмбеддингов русского языка?
*   Как составить бенчмарк? DONE, но нужно добавить синонимы
*   Как выглядят запросы? ну примерно название товаров
*   Как предобработать текст запроса? да никак, но можно попробовать расстояние ливенштейна
*   Как предобработать текст товаров? думаю не стоит ничего убирать
   
